# 7. Minimum-norm solution of underdetermined systems  [1.8.2]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.linalg import null_space
from ipywidgets import interact, FloatSlider

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

def line3d(P, color, name, width=5, dash=None):
    return go.Scatter3d(x=P[0], y=P[1], z=P[2], mode="lines", name=name,
                        line=dict(color=color, width=width, dash=dash))

def point3d(p, color, name, size=6):
    return go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]], mode="markers", name=name,
                        marker=dict(color=color, size=size))

---
## 7.1 One equation, two unknowns

**Task.** Plot the solution line of $2x_1 + x_2 = 3$, circles $\|x\|_2 = r$ with a slider for $r$, and the point

$$\min_x \|x\|_2^2 \ \text{ s.t. } Ax = b \quad\Longrightarrow\quad x_r = A^\top(AA^\top)^{-1}b.$$

Here $A = [2\ \ 1]$ (one row) and $b = 3$. Also drawn: the row space (dotted, direction $[2, 1]$) and the null-space direction $[1, -2]$.

### Prediction (write this *before* running)

- At what radius $r^*$ does the circle first touch the line? Where?

*(fill in)*

In [ ]:
A = np.array([[2.0, 1.0]])
b = np.array([3.0])
x_r = A.T @ np.linalg.solve(A @ A.T, b)
x_r = x_r.ravel()
r_star = np.linalg.norm(x_r)
n_dir = np.array([1.0, -2.0]) / np.sqrt(5)                  # unit null-space direction
print("x_r =", x_r, "  ‖x_r‖ =", r_star)

def plot_circle(r):
    fig, ax = plt.subplots(figsize=(7, 7))
    s = np.linspace(-6, 6, 2)
    ax.plot(*(x_r[:, None] + np.outer(n_dir, s)), "k", lw=2, label="2x₁ + x₂ = 3")
    ax.plot(*np.outer([2, 1], [-2, 2]), ":", color="royalblue", label="row space  span{[2, 1]}")
    ax.plot(*np.outer(n_dir, [-4, 4]), ":", color="crimson", label="null space  span{[1, −2]}")
    touches = r >= r_star - 1e-9
    ax.add_patch(plt.Circle((0, 0), r, fill=False, lw=2, color="seagreen" if touches else "gray"))
    if touches:                                               # |x_r + s n|² = r²  ⇒  s = ±√(r² − ‖x_r‖²)
        for sgn in [-1, 1]:
            ax.plot(*(x_r + sgn * np.sqrt(max(r**2 - r_star**2, 0)) * n_dir), "o", color="seagreen", ms=8)
    ax.annotate("", xy=x_r, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="darkorange", lw=2.5))
    ax.plot(*x_r, "o", color="darkorange", ms=9, label=f"x_r = {x_r.round(3)}")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.axhline(0, c="k", lw=0.5); ax.axvline(0, c="k", lw=0.5); ax.legend(loc="lower left")
    n_pts = 0 if not touches else (1 if np.isclose(r, r_star, atol=5e-3) else 2)
    ax.set_title(f"r = {r:.3f}   (‖x_r‖ = {r_star:.4f})   →  ~{n_pts} intersection point(s)")
    plt.show()

@interact(r=FloatSlider(min=0.2, max=3, step=0.01, value=0.8))
def explore_circle(r):
    plot_circle(r)

### Verify numerically

In [ ]:
check("A x_r = b", np.allclose(A @ x_r, b))
check("x_r = [1.2, 0.6]", np.allclose(x_r, [1.2, 0.6]))
check("x_r ∥ [2, 1]  (it is in the row space)", np.isclose(x_r[0] * 1 - x_r[1] * 2, 0))
check("x_r ⟂ null-space direction", np.isclose(x_r @ n_dir, 0))
check("‖x_r‖ = |b| / ‖a‖ = 3/√5", np.isclose(r_star, 3 / np.sqrt(5)))
check("same as pinv and lstsq", np.allclose(np.linalg.pinv(A) @ b, x_r) and np.allclose(np.linalg.lstsq(A, b, rcond=None)[0], x_r))
others = x_r[:, None] + np.outer(n_dir, np.random.default_rng(0).normal(0, 1, 1000))
check("1000 other solutions all have a larger norm", np.all(np.linalg.norm(others, axis=0) > r_star))

---
### Explain (hand-in — in your own words)

**1. Why is the touching point the minimum-norm solution?**

*Your answer.*

**2. Why is it perpendicular to the line (i.e. in the row space)?**
*(Hint: every solution is $x_r + s\,n$ with $n$ in the null space. At the touching point the circle is tangent to the line, so the radius is perpendicular to it.)*

*Your answer.*

---
## 7.2 Reproduce Example 1.13

**Task.** Compute $x_r$ and the null-space vector, and plot in 3-D the solution line $x_r + c\,x_{ns}$.

$$\begin{bmatrix}2&1&0\\1&3&1\end{bmatrix}x = \begin{bmatrix}3\\1\end{bmatrix}, \qquad x_r = \frac{1}{30}\begin{bmatrix}46\\-2\\-10\end{bmatrix}, \qquad x_{ns} = \begin{bmatrix}1\\-2\\5\end{bmatrix}.$$

### Prediction (write this *before* running)

- Which point of the solution line is closest to the origin? What is $g(c)$ there?

*(fill in)*

In [ ]:
A = np.array([[2.0, 1.0, 0.0], [1.0, 3.0, 1.0]])
b = np.array([3.0, 1.0])
x_r = A.T @ np.linalg.solve(A @ A.T, b)
x_ns = np.array([1.0, -2.0, 5.0])
ns_unit = null_space(A)[:, 0]
print("x_r      =", x_r, "  = 1/30 ·", 30 * x_r)
print("null_space(A) =", ns_unit, "  rescaled:", ns_unit / ns_unit[0])

In [ ]:
check("x_r = 1/30 [46, −2, −10]", np.allclose(x_r, np.array([46, -2, -10]) / 30))
check("(AAᵀ)⁻¹ = [[11/30, −1/6], [−1/6, 1/6]]  (notes)", np.allclose(np.linalg.inv(A @ A.T), [[11/30, -1/6], [-1/6, 1/6]]))
check("A x_r = b", np.allclose(A @ x_r, b))
check("A x_ns = 0", np.allclose(A @ x_ns, 0))
check("scipy's null_space is parallel to x_ns", np.allclose(ns_unit / ns_unit[0], x_ns))
check("x_rᵀ x_ns = 0", np.isclose(x_r @ x_ns, 0))
check("x_r = pinv(A) b = lstsq solution (minimum norm)",
      np.allclose(np.linalg.pinv(A) @ b, x_r) and np.allclose(np.linalg.lstsq(A, b, rcond=None)[0], x_r))
check("x_r + c x_ns solves Ax = b for c = 0.7", np.allclose(A @ (x_r + 0.7 * x_ns), b))

In [ ]:
cs = np.linspace(-0.6, 0.6, 2)
sol_line = x_r[:, None] + np.outer(x_ns, cs)
_, _, Vt = np.linalg.svd(A)
g = np.linspace(-2.5, 2.5, 2)
G1, G2 = np.meshgrid(g, g)
RS = Vt[[0]].T * G1.ravel() + Vt[[1]].T * G2.ravel()                    # row-space plane
o = np.zeros(3)
samples = [x_r + c * x_ns for c in [-0.4, -0.2, 0.2, 0.4]]

fig = go.Figure([
    go.Surface(x=RS[0].reshape(2, 2), y=RS[1].reshape(2, 2), z=RS[2].reshape(2, 2), opacity=0.25,
               colorscale=[[0, "royalblue"], [1, "royalblue"]], showscale=False, name="RS(A)", showlegend=True),
    line3d(sol_line, "black", "solution line x_r + c x_ns", width=6),
    line3d(np.column_stack([o, x_r]), "darkorange", "x_r", width=8),
    line3d(np.column_stack([x_r, x_r + 0.4 * x_ns]), "crimson", "0.4 · x_ns", width=8),
    point3d(o, "black", "origin"),
    point3d(x_r, "darkorange", f"x_r  (‖x_r‖ = {np.linalg.norm(x_r):.3f})", 8),
] + [point3d(p, "gray", f"c = {c}: ‖x‖ = {np.linalg.norm(p):.3f}", 5)
     for c, p in zip([-0.4, -0.2, 0.2, 0.4], samples)])
fig.update_layout(scene=dict(aspectmode="data"), height=600)
fig.show()

### $g(c) = \|x_r + c\,x_{ns}\|_2^2 = \|x_r\|_2^2 + c^2\|x_{ns}\|_2^2$

In [ ]:
c = np.linspace(-1, 1, 201)
g_num = np.array([np.sum((x_r + ci * x_ns) ** 2) for ci in c])
g_formula = x_r @ x_r + c**2 * (x_ns @ x_ns)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(c, g_num, lw=3, label="‖x_r + c x_ns‖²  (computed)")
ax.plot(c, g_formula, "k--", label="‖x_r‖² + c²‖x_ns‖²")
ax.plot(0, x_r @ x_r, "r*", ms=15, label=f"minimum at c = 0: {x_r @ x_r:.4f}")
ax.set_xlabel("c"); ax.set_ylabel("g(c)"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
check("g(c) = ‖x_r‖² + c²‖x_ns‖² for all c (no cross term)", np.allclose(g_num, g_formula))

---
### Explain (hand-in — in your own words)

**1. Check $x_r^\top x_{ns} = 0$. Why must it be zero?**
*(Hint: $x_r$ is in the row space.)*

*Your answer.*

**2. Plot $g(c)$: where is its minimum, and why?**
*(Hint: expand $\|x_r + c\,x_{ns}\|^2$. Which term vanishes?)*

*Your answer.*

---
## 7.3 Gradient descent finds it too

**Task.** Run gradient descent from $x_0 = 0$ for a wide $A$, and plot the iterates in 3-D.

$$x_{k+1} = x_k - \eta\, A^\top(Ax_k - b).$$

We use $A$, $b$ from Example 1.13 and step size $\eta = 1/\sigma_{\max}^2$. A second run starts from a non-zero $x_0$.

### Prediction (write this *before* running)

- Where does gradient descent end up from $x_0 = 0$? From another $x_0$?

*(fill in)*

In [ ]:
def gradient_descent(A, b, x0, eta, iters=200):
    xs = [np.array(x0, dtype=float)]
    for _ in range(iters):
        xs.append(xs[-1] - eta * A.T @ (A @ xs[-1] - b))
    return np.array(xs)

sigma = np.linalg.svd(A, compute_uv=False)
eta = 1 / sigma[0] ** 2
n_unit = ns_unit / np.linalg.norm(ns_unit)
P_NS = np.outer(n_unit, n_unit)                              # projector onto the null space

x0_alt = np.array([1.5, 1.0, -1.0])
X_zero = gradient_descent(A, b, np.zeros(3), eta)
X_alt = gradient_descent(A, b, x0_alt, eta)
x_lim_pred = x_r + P_NS @ x0_alt
print("from 0      → ", X_zero[-1], "   x_r =", x_r)
print("from x0_alt → ", X_alt[-1], "   x_r + P_NS x0 =", x_lim_pred)

In [ ]:
sol_line = x_r[:, None] + np.outer(x_ns, np.linspace(-0.6, 0.6, 2))
fig = go.Figure([
    line3d(sol_line, "black", "solution set", width=6),
    go.Scatter3d(x=X_zero[:40, 0], y=X_zero[:40, 1], z=X_zero[:40, 2], mode="lines+markers", name="GD from 0",
                 line=dict(color="royalblue"), marker=dict(size=3)),
    go.Scatter3d(x=X_alt[:40, 0], y=X_alt[:40, 1], z=X_alt[:40, 2], mode="lines+markers", name="GD from x0_alt",
                 line=dict(color="darkorange"), marker=dict(size=3)),
    point3d(x_r, "red", "x_r (minimum norm)", 8),
    point3d(x_lim_pred, "darkorange", "x_r + P_NS x0", 8),
    point3d(np.zeros(3), "black", "origin"),
])
fig.update_layout(scene=dict(aspectmode="data"), height=600)
fig.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
for X, col, lab, target in [(X_zero, "royalblue", "from 0", x_r), (X_alt, "darkorange", "from x0_alt", x_lim_pred)]:
    ax1.semilogy(np.linalg.norm(X - target, axis=1) + 1e-17, color=col, label=lab)
    ax2.plot(X @ n_unit, color=col, label=lab)
ax1.set_xlabel("iteration"); ax1.set_ylabel("distance to its limit"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_xlabel("iteration"); ax2.set_ylabel("null-space component  nᵀx_k"); ax2.legend(); ax2.grid(alpha=0.3)
plt.show()

### Verify numerically

In [ ]:
updates = np.diff(X_zero, axis=0)
check("every update −ηAᵀ(Ax−b) is in the row space (no null-space component)", np.allclose(updates @ n_unit, 0))
check("all iterates from x₀ = 0 stay in the row space", np.allclose(X_zero @ n_unit, 0))
check("GD from 0 converges to x_r", np.allclose(X_zero[-1], x_r))
check("from x0_alt the null-space component never changes", np.allclose(X_alt @ n_unit, x0_alt @ n_unit))
check("GD from x0_alt converges to x_r + P_NS x0", np.allclose(X_alt[-1], x_lim_pred))

### Explore: the step size. Predict first: what happens for $\eta > 2/\sigma_{\max}^2$?

In [ ]:
@interact(eta_factor=FloatSlider(min=0.1, max=2.2, step=0.05, value=1.0, description="η·σ²max"))
def explore_step(eta_factor):
    X = gradient_descent(A, b, np.zeros(3), eta_factor / sigma[0] ** 2, iters=100)
    plt.figure(figsize=(8, 4))
    plt.semilogy(np.linalg.norm(X - x_r, axis=1) + 1e-17)
    plt.xlabel("iteration"); plt.ylabel("‖x_k − x_r‖"); plt.grid(alpha=0.3)
    plt.title(f"η = {eta_factor:.2f} / σ²max")
    plt.show()

---
### Explain (hand-in — in your own words)

**1. Every update lies in the row space of $A$. Why does this make it converge to the minimum-norm solution?**
*(Hint: starting from 0, which subspace can the iterates never leave? Which solution lives in that subspace?)*

*Your answer.*

**2. What happens if you start from a nonzero $x_0$?**
*(Hint: split $x_0$ into its row-space and null-space parts.)*

*Your answer.*